# Avoided EAD elevation analysis

This notebook tests whether infrastructure locations benefiting from nature-based landslide risk reduction occur at higher elevations than infrastructure locations benefiting from coastal flood and river flood risk reduction. It uses the cross-cutting Jamaica DEM and summarises elevations both unweighted and weighted by avoided EAD.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import Resampling, reproject


def find_project_root(start_path: Path | None = None) -> Path:
    search_start = Path.cwd() if start_path is None else Path(start_path)
    for candidate_path in [search_start, *search_start.parents]:
        if (candidate_path / "dphil_papers").exists():
            return candidate_path
    raise FileNotFoundError("Could not find project root containing dphil_papers")


ROOT = find_project_root()
PAPERS = ROOT / "dphil_papers"
PAPER2 = PAPERS / "dphil_paper_2"
PAPER3 = PAPERS / "dphil_paper_3"
COMMON = PAPERS / "dphil_common_cross_cutting"

CRS_METRIC = "EPSG:3448"
JMD_TO_USD = 1.0 / 150.0
RIVER_MINIMUM_POSITIVE_USD = JMD_TO_USD

DEM_PATH = COMMON / "common_processed_data" / "processed_dem" / "jamaica_dem_3448.tif"
COASTAL_ASSET_LOCATIONS_PATH = PAPER3 / "results" / "03_cross_hazard_comparison" / "spatial_service_provision_comparison" / "coastal_avoided_ead_mangrove_locations" / "coastal_positive_avoided_asset_locations_maximum.geoparquet"
RIVER_AVOIDED_EAD_RASTER_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "avoided_fluvial_eads" / "avoided__fluvial__ead_max_300m_smoothed.tif"
LANDSLIDE_ASSET_MAP_LAYERS_PATH = PAPER3 / "results" / "02_damage_estimates" / "landslide_damages" / "results_landslide_maximum_scenario_combined_class" / "damage_estimates" / "landslide_source_and_runout_ead_asset_map_layers_combined_class.gpkg"
LANDSLIDE_LAYER_NAME = "landslide_source_and_runout_ead_asset_map_layers_combined_class"

OUTPUT_DIR = PAPER3 / "results" / "03_cross_hazard_comparison" / "spatial_service_provision_comparison" / "elevation_check"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LANDSLIDE_COLUMNS = [
    "Sector",
    "Subsector",
    "Avoided_EAD_Protection_USD",
    "Avoided_EAD_Reafforestation_USD",
    "geometry",
]
IMPORTANT_SUBSECTORS = [
    "roads",
    "rail lines",
    "ports",
    "buildings",
    "electricity nodes",
    "potable pipelines",
]
MAIN_COMPARISON_GROUPS = [
    "Jamaica DEM valid land cells",
    "Coastal flood positive avoided-EAD infrastructure locations",
    "River flood positive avoided-EAD infrastructure raster cells",
    "Landslide forest protection positive avoided-EAD infrastructure locations",
    "Landslide forest restoration positive avoided-EAD infrastructure locations",
    "Landslide forest protection positive avoided-EAD infrastructure locations: roads",
    "Landslide forest restoration positive avoided-EAD infrastructure locations: roads",
]
PLOT_LABELS = {
    "Coastal flood positive avoided-EAD infrastructure locations": "Coastal flood",
    "River flood positive avoided-EAD infrastructure raster cells": "River flood",
    "Landslide forest protection positive avoided-EAD infrastructure locations": "Landslide protection",
    "Landslide forest restoration positive avoided-EAD infrastructure locations": "Landslide restoration",
    "Landslide forest protection positive avoided-EAD infrastructure locations: roads": "Protection roads",
    "Landslide forest restoration positive avoided-EAD infrastructure locations: roads": "Restoration roads",
}
PLOT_COLORS = {
    "Coastal flood": "#2b8cbe",
    "River flood": "#41ab5d",
    "Landslide protection": "#756bb1",
    "Landslide restoration": "#9e9ac8",
    "Protection roads": "#d95f0e",
    "Restoration roads": "#fdae6b",
}
NATURE_RC = {
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "figure.titlesize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}

for required_input_path in [
    DEM_PATH,
    COASTAL_ASSET_LOCATIONS_PATH,
    RIVER_AVOIDED_EAD_RASTER_PATH,
    LANDSLIDE_ASSET_MAP_LAYERS_PATH,
]:
    if not required_input_path.exists():
        raise FileNotFoundError(required_input_path)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
OUTPUT_DIR

## Helper functions

For vector infrastructure layers, elevation is sampled from the DEM at each feature's representative point. This gives one elevation per asset feature, including line and polygon features. For the river-flood raster, the DEM is reprojected to the river avoided-EAD grid and summarised for positive avoided-EAD cells.

In [ ]:
def weighted_quantile(values: np.ndarray, probabilities: list[float], weights: np.ndarray | None = None) -> np.ndarray:
    value_array = np.asarray(values, dtype="float64")
    probability_array = np.asarray(probabilities, dtype="float64")
    finite_value_mask = np.isfinite(value_array)

    if weights is None:
        return np.nanquantile(value_array[finite_value_mask], probability_array)

    weight_array = np.asarray(weights, dtype="float64")
    finite_weight_mask = finite_value_mask & np.isfinite(weight_array) & (weight_array > 0)
    if not finite_weight_mask.any():
        return np.full(len(probability_array), np.nan, dtype="float64")

    valid_values = value_array[finite_weight_mask]
    valid_weights = weight_array[finite_weight_mask]
    sort_order = np.argsort(valid_values)
    sorted_values = valid_values[sort_order]
    sorted_weights = valid_weights[sort_order]
    cumulative_probability = (np.cumsum(sorted_weights) - 0.5 * sorted_weights) / np.sum(sorted_weights)
    return np.interp(probability_array, cumulative_probability, sorted_values)


def summarise_elevations(
    group_name: str,
    source_type: str,
    elevations_m: np.ndarray,
    weights_usd: np.ndarray | None = None,
) -> dict:
    elevation_array = np.asarray(elevations_m, dtype="float64")
    valid_elevation_mask = np.isfinite(elevation_array)
    valid_elevations = elevation_array[valid_elevation_mask]

    summary_row = {
        "group": group_name,
        "source_type": source_type,
        "observation_count": int(valid_elevation_mask.sum()),
        "mean_m": float(np.nanmean(valid_elevations)) if len(valid_elevations) else np.nan,
        "p10_m": float(np.nanquantile(valid_elevations, 0.10)) if len(valid_elevations) else np.nan,
        "median_m": float(np.nanquantile(valid_elevations, 0.50)) if len(valid_elevations) else np.nan,
        "p90_m": float(np.nanquantile(valid_elevations, 0.90)) if len(valid_elevations) else np.nan,
        "share_above_200m": float(np.mean(valid_elevations >= 200)) if len(valid_elevations) else np.nan,
        "share_above_500m": float(np.mean(valid_elevations >= 500)) if len(valid_elevations) else np.nan,
    }

    if weights_usd is None:
        return summary_row

    weight_array = np.asarray(weights_usd, dtype="float64")
    valid_weight_mask = valid_elevation_mask & np.isfinite(weight_array) & (weight_array > 0)
    if not valid_weight_mask.any():
        return summary_row

    weighted_quantiles = weighted_quantile(
        elevation_array[valid_weight_mask],
        [0.10, 0.50, 0.90],
        weight_array[valid_weight_mask],
    )
    total_weight = float(np.sum(weight_array[valid_weight_mask]))
    summary_row.update(
        {
            "weighted_observation_count": int(valid_weight_mask.sum()),
            "weight_total_usd": total_weight,
            "weighted_mean_m": float(np.average(elevation_array[valid_weight_mask], weights=weight_array[valid_weight_mask])),
            "weighted_p10_m": float(weighted_quantiles[0]),
            "weighted_median_m": float(weighted_quantiles[1]),
            "weighted_p90_m": float(weighted_quantiles[2]),
            "weighted_share_above_200m": float(
                np.sum(weight_array[valid_weight_mask & (elevation_array >= 200)]) / total_weight
            ),
            "weighted_share_above_500m": float(
                np.sum(weight_array[valid_weight_mask & (elevation_array >= 500)]) / total_weight
            ),
        }
    )
    return summary_row


def sample_dem_at_representative_points(geodataframe: gpd.GeoDataFrame, dem_dataset: rasterio.io.DatasetReader) -> np.ndarray:
    metric_geodataframe = geodataframe if geodataframe.crs == dem_dataset.crs else geodataframe.to_crs(dem_dataset.crs)
    representative_points = metric_geodataframe.geometry.representative_point()
    sample_coordinates = [(point.x, point.y) for point in representative_points]
    sampled_values = np.array([sample[0] for sample in dem_dataset.sample(sample_coordinates)], dtype="float64")
    if dem_dataset.nodata is not None:
        sampled_values[sampled_values == dem_dataset.nodata] = np.nan
    sampled_values[sampled_values < -100] = np.nan
    return sampled_values


def extract_river_elevations_and_weights(
    river_raster_path: Path,
    dem_dataset: rasterio.io.DatasetReader,
) -> tuple[np.ndarray, np.ndarray]:
    with rasterio.open(river_raster_path) as river_dataset:
        river_values_masked = river_dataset.read(1, masked=True)
        river_avoided_ead_usd = np.array(river_values_masked, dtype="float64") * JMD_TO_USD
        if np.ma.is_masked(river_values_masked):
            river_avoided_ead_usd[river_values_masked.mask] = np.nan
        river_avoided_ead_usd[river_avoided_ead_usd <= RIVER_MINIMUM_POSITIVE_USD] = np.nan

        dem_on_river_grid = np.full((river_dataset.height, river_dataset.width), np.nan, dtype="float32")
        reproject(
            source=rasterio.band(dem_dataset, 1),
            destination=dem_on_river_grid,
            src_transform=dem_dataset.transform,
            src_crs=dem_dataset.crs,
            src_nodata=dem_dataset.nodata,
            dst_transform=river_dataset.transform,
            dst_crs=river_dataset.crs,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )

    valid_river_mask = (
        np.isfinite(river_avoided_ead_usd)
        & (river_avoided_ead_usd > RIVER_MINIMUM_POSITIVE_USD)
        & np.isfinite(dem_on_river_grid)
        & (dem_on_river_grid > -100)
    )
    return dem_on_river_grid[valid_river_mask], river_avoided_ead_usd[valid_river_mask]


def append_sector_and_subsector_summaries(
    summary_rows: list[dict],
    base_group_name: str,
    geodataframe: gpd.GeoDataFrame,
    weight_column: str,
) -> None:
    summary_rows.append(
        summarise_elevations(
            base_group_name,
            "overall",
            geodataframe["elevation_m"].to_numpy(),
            geodataframe[weight_column].to_numpy(),
        )
    )

    for sector_name, sector_data in geodataframe.groupby("Sector", sort=True):
        summary_rows.append(
            summarise_elevations(
                f"{base_group_name}: {sector_name}",
                "sector",
                sector_data["elevation_m"].to_numpy(),
                sector_data[weight_column].to_numpy(),
            )
        )

    for subsector_name in IMPORTANT_SUBSECTORS:
        subsector_data = geodataframe.loc[geodataframe["Subsector"] == subsector_name]
        if len(subsector_data) == 0:
            continue
        summary_rows.append(
            summarise_elevations(
                f"{base_group_name}: {subsector_name}",
                "subsector",
                subsector_data["elevation_m"].to_numpy(),
                subsector_data[weight_column].to_numpy(),
            )
        )


def rounded_display(dataframe: pd.DataFrame) -> pd.DataFrame:
    rounded_dataframe = dataframe.copy()
    numeric_columns = rounded_dataframe.select_dtypes(include="number").columns
    rounded_dataframe[numeric_columns] = rounded_dataframe[numeric_columns].round(3)
    return rounded_dataframe

## Load data and sample elevations

In [ ]:
positive_coastal_asset_locations = gpd.read_parquet(COASTAL_ASSET_LOCATIONS_PATH).to_crs(CRS_METRIC)
landslide_asset_locations = gpd.read_file(
    LANDSLIDE_ASSET_MAP_LAYERS_PATH,
    layer=LANDSLIDE_LAYER_NAME,
    columns=LANDSLIDE_COLUMNS,
).to_crs(CRS_METRIC)
landslide_positive_locations = landslide_asset_locations.loc[
    (landslide_asset_locations["Avoided_EAD_Protection_USD"] > 0)
    | (landslide_asset_locations["Avoided_EAD_Reafforestation_USD"] > 0)
].copy()

with rasterio.open(DEM_PATH) as dem_dataset:
    dem_metadata = pd.DataFrame(
        [
            {
                "path": DEM_PATH,
                "crs": str(dem_dataset.crs),
                "width": dem_dataset.width,
                "height": dem_dataset.height,
                "x_resolution_m": dem_dataset.res[0],
                "y_resolution_m": dem_dataset.res[1],
                "nodata": dem_dataset.nodata,
            }
        ]
    )
    dem_array = dem_dataset.read(1, masked=True).astype("float64")
    dem_valid_elevations = dem_array.compressed()
    positive_coastal_asset_locations["elevation_m"] = sample_dem_at_representative_points(
        positive_coastal_asset_locations,
        dem_dataset,
    )
    landslide_positive_locations["elevation_m"] = sample_dem_at_representative_points(
        landslide_positive_locations,
        dem_dataset,
    )
    river_elevations_m, river_avoided_ead_usd = extract_river_elevations_and_weights(
        RIVER_AVOIDED_EAD_RASTER_PATH,
        dem_dataset,
    )

positive_landslide_protection_locations = landslide_positive_locations.loc[
    landslide_positive_locations["Avoided_EAD_Protection_USD"] > 0
].copy()
positive_landslide_restoration_locations = landslide_positive_locations.loc[
    landslide_positive_locations["Avoided_EAD_Reafforestation_USD"] > 0
].copy()

dem_metadata

## Elevation summaries

In [ ]:
summary_rows = [
    summarise_elevations(
        "Jamaica DEM valid land cells",
        "baseline",
        dem_valid_elevations,
    ),
    summarise_elevations(
        "Coastal flood positive avoided-EAD infrastructure locations",
        "overall",
        positive_coastal_asset_locations["elevation_m"].to_numpy(),
        positive_coastal_asset_locations["Avoided_EAD_USD"].to_numpy(),
    ),
    summarise_elevations(
        "River flood positive avoided-EAD infrastructure raster cells",
        "overall",
        river_elevations_m,
        river_avoided_ead_usd,
    ),
]

for coastal_sector_name, coastal_sector_data in positive_coastal_asset_locations.groupby("Sector", sort=True):
    summary_rows.append(
        summarise_elevations(
            f"Coastal flood positive avoided-EAD infrastructure locations: {coastal_sector_name}",
            "sector",
            coastal_sector_data["elevation_m"].to_numpy(),
            coastal_sector_data["Avoided_EAD_USD"].to_numpy(),
        )
    )

append_sector_and_subsector_summaries(
    summary_rows,
    "Landslide forest protection positive avoided-EAD infrastructure locations",
    positive_landslide_protection_locations,
    "Avoided_EAD_Protection_USD",
)
append_sector_and_subsector_summaries(
    summary_rows,
    "Landslide forest restoration positive avoided-EAD infrastructure locations",
    positive_landslide_restoration_locations,
    "Avoided_EAD_Reafforestation_USD",
)

elevation_summary = pd.DataFrame(summary_rows)
elevation_summary.to_csv(OUTPUT_DIR / "avoided_ead_elevation_summary.csv", index=False)

main_comparison = elevation_summary.set_index("group").loc[MAIN_COMPARISON_GROUPS].reset_index()
main_comparison.to_csv(OUTPUT_DIR / "avoided_ead_elevation_main_comparison.csv", index=False)

rounded_display(main_comparison)[
    [
        "group",
        "observation_count",
        "median_m",
        "share_above_200m",
        "share_above_500m",
        "weighted_median_m",
        "weighted_share_above_200m",
        "weighted_share_above_500m",
        "weight_total_usd",
    ]
]

## Sector and subsector detail

The table below is useful for checking whether the elevation signal holds for roads and other important infrastructure subsectors, rather than only for the combined landslide layer.

In [ ]:
sector_subsector_summary = elevation_summary.loc[elevation_summary["source_type"].isin(["sector", "subsector"])].copy()
sector_subsector_summary.to_csv(OUTPUT_DIR / "avoided_ead_elevation_sector_subsector_summary.csv", index=False)

rounded_display(sector_subsector_summary.loc[
    sector_subsector_summary["group"].str.contains("Landslide|Coastal"),
    [
        "group",
        "source_type",
        "observation_count",
        "median_m",
        "share_above_200m",
        "share_above_500m",
        "weighted_median_m",
        "weighted_share_above_200m",
        "weighted_share_above_500m",
        "weight_total_usd",
    ],
])

## Comparison figure

In [ ]:
plot_comparison = main_comparison.loc[main_comparison["group"].isin(PLOT_LABELS)].copy()
plot_comparison["plot_label"] = plot_comparison["group"].map(PLOT_LABELS)
plot_comparison["plot_color"] = plot_comparison["plot_label"].map(PLOT_COLORS)
plot_comparison["weighted_share_above_200_percent"] = plot_comparison["weighted_share_above_200m"] * 100
plot_comparison["weighted_share_above_500_percent"] = plot_comparison["weighted_share_above_500m"] * 100
plot_comparison = plot_comparison.iloc[::-1].reset_index(drop=True)

with mpl.rc_context(NATURE_RC):
    figure, axes = plt.subplots(1, 2, figsize=(7.2, 3.2), constrained_layout=True)

    axes[0].barh(
        plot_comparison["plot_label"],
        plot_comparison["weighted_median_m"],
        color=plot_comparison["plot_color"],
    )
    axes[0].set_xlabel("EAD-weighted median elevation (m)")
    axes[0].set_ylabel("")
    axes[0].grid(axis="x", color="#d9d9d9", linewidth=0.4)
    axes[0].set_axisbelow(True)

    axes[1].barh(
        plot_comparison["plot_label"],
        plot_comparison["weighted_share_above_200_percent"],
        color=plot_comparison["plot_color"],
    )
    axes[1].set_xlabel("Avoided EAD above 200 m (%)")
    axes[1].set_ylabel("")
    axes[1].set_xlim(0, 100)
    axes[1].grid(axis="x", color="#d9d9d9", linewidth=0.4)
    axes[1].set_axisbelow(True)

    for axis in axes:
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)

    figure.savefig(OUTPUT_DIR / "avoided_ead_elevation_comparison.png", dpi=300, bbox_inches="tight")
    figure.savefig(OUTPUT_DIR / "avoided_ead_elevation_comparison.pdf", bbox_inches="tight")
    plt.show()

## Manuscript-facing interpretation check

The strongest defensible statement is not that all landslide benefits occur in uplands. Instead, the DEM supports saying that landslide avoided damages occur at substantially higher elevations than coastal-flood benefits, and that this elevation signal is strongest for forest restoration and for road benefits under restoration.

In [ ]:
metric_lookup = main_comparison.set_index("group")
coastal_weighted_median = metric_lookup.loc[
    "Coastal flood positive avoided-EAD infrastructure locations",
    "weighted_median_m",
]
river_weighted_median = metric_lookup.loc[
    "River flood positive avoided-EAD infrastructure raster cells",
    "weighted_median_m",
]
protection_weighted_median = metric_lookup.loc[
    "Landslide forest protection positive avoided-EAD infrastructure locations",
    "weighted_median_m",
]
restoration_weighted_median = metric_lookup.loc[
    "Landslide forest restoration positive avoided-EAD infrastructure locations",
    "weighted_median_m",
]
protection_roads_weighted_median = metric_lookup.loc[
    "Landslide forest protection positive avoided-EAD infrastructure locations: roads",
    "weighted_median_m",
]
restoration_roads_weighted_median = metric_lookup.loc[
    "Landslide forest restoration positive avoided-EAD infrastructure locations: roads",
    "weighted_median_m",
]

print(
    "Suggested interpretation:\n"
    f"Coastal flood avoided-EAD locations have an EAD-weighted median elevation of {coastal_weighted_median:.0f} m. "
    f"River-flood avoided-EAD raster cells have an EAD-weighted median elevation of {river_weighted_median:.0f} m. "
    f"Landslide avoided-EAD locations are higher, with EAD-weighted medians of {protection_weighted_median:.0f} m under forest protection "
    f"and {restoration_weighted_median:.0f} m under forest restoration. "
    f"For roads, the corresponding EAD-weighted medians are {protection_roads_weighted_median:.0f} m and "
    f"{restoration_roads_weighted_median:.0f} m. This supports describing landslide benefits as concentrated in inland and "
    "higher-elevation infrastructure locations, especially for forest restoration, rather than claiming all benefits occur in upland corridors."
)